# Wind resource layers — multi-variable windowed read (live)

One `download()` call can fetch **several** Global Wind Atlas layers for the same
bounding box. Each is read **windowed** from its remote Cloud-Optimized GeoTIFF
(only the AOI transfers), and `download()` returns one written GeoTIFF per layer.
Here we pull the mean wind speed, the IEC-class-2 turbine capacity factor, and the
air density at 100 m for a small box near DTU (Denmark) and compare them.

_Runs live against public figshare (no credentials)._


In [ ]:
import numpy as np
from pyramids.dataset import Dataset

from earthlens.core import EarthLens

## Fetch three wind layers at once

Pass several layer ids in `variables=`; the backend resolves each against the
catalog and fetches it by its transport (all three here are `vsicurl` windowed
reads). The returned list has one GeoTIFF path per requested layer, in order.


In [ ]:
layers = ["wind_100m", "capacity_factor_iec2", "air_density_100m"]
paths = EarthLens(
    data_source="solar-wind-atlas",
    variables=layers,
    lat_lim=[55.0, 55.5],
    lon_lim=[12.0, 12.5],
    path="wind_layers_out",
).download(progress_bar=False)
for layer, path in zip(layers, paths):
    print(f"{layer:22s} -> {path.name}")

## Inspect each layer

Read every written GeoTIFF back with pyramids and summarise its value range. All
three share the same EPSG:4326 grid and AOI; only the quantity differs.


In [ ]:
rows = []
for layer, path in zip(layers, paths):
    ds = Dataset.read_file(path)
    a = np.asarray(ds.read_array(), dtype='float64')
    finite = a[np.isfinite(a)]
    rows.append(
        (
            layer,
            ds.columns,
            ds.rows,
            round(float(finite.min()), 3),
            round(float(finite.mean()), 3),
            round(float(finite.max()), 3),
        )
    )

import pandas as pd

pd.DataFrame(rows, columns=['layer', 'cols', 'rows', 'min', 'mean', 'max'])

## What the numbers say

- `wind_100m` is the mean wind speed at 100 m in m/s.
- `capacity_factor_iec2` is the fraction of rated output an IEC-class-2 turbine
  would average here (0-1) — it tracks the wind speed.
- `air_density_100m` (kg/m3) scales the power a turbine extracts at a given speed.

Because all three are read windowed from their global COGs, fetching three layers
for a small box still transfers only a few hundred KB each — not the multi-GB
global files. (Global Solar Atlas layers like `ghi` would instead download their
full ~2.7 GB archive once, then crop locally.)
